# WiPER Dataset Generation — v4

Legge le ampiezze prodotte da **WiPER-DataProcessing** (nessuna modifica necessaria a quel notebook) e genera i grafi PyG.

**Cosa cambia rispetto alla v3**

| # | v3 | v4 |
|---|---|---|
| 1 | Sanitizzazione fissa = **IQR** (appiattisce il movimento: su `saluto1` la std temporale scende a ~30%) | `CFG['sanitization']` = `'zscore'` (default) \| `'hampel'` \| `'iqr'` \| `'raw'` |
| 2 | Solo il primo gruppo di 100 pacchetti per file, il resto scartato (fino a ~45% della registrazione) | **Finestre scorrevoli** (`n_packets=100`, `stride=25`); `stride=100` riproduce il comportamento v3 |
| 3 | "vuoto" (stanza vuota) trattato come un'identità (finiva nel test come probe senza gallery) | Escluso (`exclude_ids`) |
| 4 | Ordine delle identità = ordine di `os.listdir` -> lo split cambiava da una macchina all'altra | Identità **ordinate** prima dello shuffle con seed -> split riproducibile |
| 5 | Il nome file conteneva solo `soggettoN_k.pt` | Ogni grafo salva `subject_id`, `action`, `rec_id`, `rec_num`, `win` (serve a valutare **per registrazione** e a costruire i protocolli stessa-azione / cross-azione) |

**Output**

* `GraphData/D1/` : tutte le finestre delle identità di training (Saluto + tPose)
* `GraphData/D2t/Gallery/` : finestre `saluto` delle identità di test
* `GraphData/D2t/Probe/` : finestre `tpose` delle identità di test

> Finestre con stride < `n_packets` si sovrappongono e provengono dalla **stessa registrazione**: nel benchmark vanno sempre aggregate per `rec_id`.

## 1. Imports e configurazione

In [8]:
import os, re, shutil, random
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data

CFG = dict(
    output_folder='./output/',
    sanitization='zscore',          # 'raw' | 'hampel' | 'zscore' | 'iqr'
    n_packets=100,                  # lunghezza finestra (= feature temporali per nodo)
    stride=25,                      # passo; 100 = finestre disgiunte (v3)
    exclude_ids=('vuoto',),         # cartelle che NON sono soggetti
    gallery_action='saluto',        # azione usata come gallery nel test
    probe_action='tpose',           # azione usata come probe nel test
    train_fraction=0.5,             # frazione di identità in D1
    seed=42,
    n_subcarriers=52,
)

SANITIZATION_DIRS = {
    'raw':    ('CSV', 'raw_amplitudes'),
    'hampel': ('CSV', 'sanitized_amplitudes', 'HF_amplitudes'),
    'zscore': ('CSV', 'sanitized_amplitudes', 'zscore_amplitudes'),
    'iqr':    ('CSV', 'sanitized_amplitudes', 'iqr_amplitudes'),
}
assert CFG['sanitization'] in SANITIZATION_DIRS, f"sanitization deve essere uno di {list(SANITIZATION_DIRS)}"
input_directory = os.path.join(CFG['output_folder'], *SANITIZATION_DIRS[CFG['sanitization']])
print('Sanitizzazione :', CFG['sanitization'])
print('Cartella input :', input_directory)

Sanitizzazione : zscore
Cartella input : ./output/CSV/sanitized_amplitudes/zscore_amplitudes


## 2. Pulizia della cartella GraphData (se esiste)

In [9]:
graph_data_dir = os.path.join(CFG['output_folder'], 'GraphData')
if os.path.exists(graph_data_dir):
    shutil.rmtree(graph_data_dir)
    print(f'Cartella {graph_data_dir} svuotata con successo!')
else:
    print(f'Nessuna cartella trovata in {graph_data_dir}, procedi pure.')

Nessuna cartella trovata in ./output/GraphData, procedi pure.


## 3. Caricamento delle ampiezze

In [10]:
KEY_RE = re.compile(r'^(?P<sid>[^_]+)_(?P<act>[A-Za-z]+)(?P<num>\d+)$')   # es. soggetto1_saluto3 / soggetto12_tPose5

def parse_key(key):
    m = KEY_RE.match(key)
    if not m:
        return None
    return m.group('sid'), m.group('act').lower(), int(m.group('num'))

assert os.path.isdir(input_directory), f'Directory non trovata: {input_directory} (hai eseguito WiPER-DataProcessing?)'

recordings = []          # una voce per file CSV (= una registrazione)
skipped_excluded, skipped_unparsed = [], []

for fname in sorted(os.listdir(input_directory)):
    if not fname.endswith('.csv'):
        continue
    key = fname[:-4]
    sid_guess = key.split('_')[0]
    if sid_guess in CFG['exclude_ids']:
        skipped_excluded.append(key); continue
    parsed = parse_key(key)
    if parsed is None:
        skipped_unparsed.append(key); continue
    sid, action, num = parsed

    df = pd.read_csv(os.path.join(input_directory, fname))
    arr = df.drop(columns='timestamp', errors='ignore').to_numpy(dtype=np.float32)    # [T, 52]
    assert arr.shape[1] == CFG['n_subcarriers'], f'{fname}: attese {CFG["n_subcarriers"]} sottoportanti, trovate {arr.shape[1]}'
    recordings.append(dict(sid=sid, action=action, num=num, rec_id=key, data=arr))

print(f'Registrazioni caricate : {len(recordings)}')
print(f'Escluse (non soggetti)  : {len(skipped_excluded)}  {sorted({k.split("_")[0] for k in skipped_excluded})}')
if skipped_unparsed:
    print(f'ATTENZIONE: nome non riconosciuto, ignorati: {skipped_unparsed[:5]}{" ..." if len(skipped_unparsed) > 5 else ""}')

actions = sorted({r['action'] for r in recordings})
print('Azioni trovate         :', actions)
for needed in (CFG['gallery_action'], CFG['probe_action']):
    assert needed in actions, f"Azione '{needed}' non trovata tra {actions}"

Registrazioni caricate : 440
Escluse (non soggetti)  : 1  ['vuoto']
Azioni trovate         : ['saluto', 'tpose']


## 4. Finestre scorrevoli

In [11]:
def sliding_windows(arr, n, stride):
    T = arr.shape[0]
    return [arr[s:s + n] for s in range(0, T - n + 1, stride)]      # lista di [n, 52]

n, stride = CFG['n_packets'], CFG['stride']
tot_packets = used_packets = 0
too_short = []
for r in recordings:
    wins = sliding_windows(r['data'], n, stride)
    r['windows'] = wins
    T = r['data'].shape[0]
    tot_packets += T
    if wins:
        last_end = (len(wins) - 1) * stride + n
        used_packets += last_end
    else:
        too_short.append((r['rec_id'], T))

n_wins = np.array([len(r['windows']) for r in recordings])
lens = np.array([r['data'].shape[0] for r in recordings])
print(f'Pacchetti per registrazione: min {lens.min()}  mediana {int(np.median(lens))}  max {lens.max()}')
print(f'Finestre per registrazione : min {n_wins.min()}  mediana {int(np.median(n_wins))}  max {n_wins.max()}  (totale {n_wins.sum()})')
v3_used = sum((r['data'].shape[0] // n) * n for r in recordings)          # copertura con finestre disgiunte (v3)
print(f'Pacchetti coperti dalle finestre: {100 * used_packets / tot_packets:.1f}%  (con finestre disgiunte come in v3: {100 * v3_used / tot_packets:.1f}%)')
if too_short:
    print(f'ATTENZIONE: {len(too_short)} registrazioni più corte di {n} pacchetti, ignorate: {too_short[:5]}')
recordings = [r for r in recordings if r['windows']]

Pacchetti per registrazione: min 100  mediana 179  max 191
Finestre per registrazione : min 1  mediana 4  max 4  (totale 1515)
Pacchetti coperti dalle finestre: 94.1%  (con finestre disgiunte come in v3: 58.4%)


## 5. Split per identità (train / test)

In [12]:
groups = defaultdict(list)
for r in recordings:
    groups[r['sid']].append(r)

def natural_key(s):
    return [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', s)]

all_identities = sorted(groups.keys(), key=natural_key)      # ordinati -> split riproducibile ovunque
rng = random.Random(CFG['seed'])
rng.shuffle(all_identities)

n_train = int(round(len(all_identities) * CFG['train_fraction']))
train_ids = all_identities[:n_train]
test_ids  = all_identities[n_train:]
print(f'Soggetti totali: {len(all_identities)} | train: {len(train_ids)} | test: {len(test_ids)}')

# controllo: ogni soggetto deve avere entrambe le azioni
for sid in all_identities:
    acts = {r['action'] for r in groups[sid]}
    if not {CFG['gallery_action'], CFG['probe_action']} <= acts:
        print(f'  ATTENZIONE: {sid} non ha entrambe le azioni ({sorted(acts)})')
print('Registrazioni per soggetto:', sorted({len(v) for v in groups.values()}))

Soggetti totali: 44 | train: 22 | test: 22
Registrazioni per soggetto: [10]


## 6. Salvataggio dei grafi

In [13]:
num_nodes = CFG['n_subcarriers']
row_idx, col_idx = torch.meshgrid(torch.arange(num_nodes), torch.arange(num_nodes), indexing='ij')
FULL_EDGE_INDEX = torch.stack([row_idx.flatten(), col_idx.flatten()], dim=0).long()   # grafo completo (come v3)

def save_graph(window, rec, win_idx, out_dir):
    x = torch.tensor(window.T, dtype=torch.float32)                  # [52 nodi, 100 feature temporali]
    g = Data(x=x, edge_index=FULL_EDGE_INDEX,
             subject_id=rec['sid'], action=rec['action'], rec_id=rec['rec_id'],
             rec_num=rec['num'], win=win_idx)
    os.makedirs(out_dir, exist_ok=True)
    torch.save(g, os.path.join(out_dir, f"{rec['rec_id']}_w{win_idx}.pt"))

root = os.path.join(CFG['output_folder'], 'GraphData')
counts = defaultdict(int)

for sid in train_ids:                                                # D1: tutte le azioni
    for rec in groups[sid]:
        for w, win in enumerate(rec['windows']):
            save_graph(win, rec, w, os.path.join(root, 'D1')); counts['D1'] += 1

for sid in test_ids:                                                 # D2t: gallery = saluto, probe = tPose
    for rec in groups[sid]:
        if rec['action'] == CFG['gallery_action']:
            sub = 'Gallery'
        elif rec['action'] == CFG['probe_action']:
            sub = 'Probe'
        else:
            continue
        for w, win in enumerate(rec['windows']):
            save_graph(win, rec, w, os.path.join(root, 'D2t', sub)); counts[f'D2t/{sub}'] += 1

print('Grafi salvati:', dict(counts))

Grafi salvati: {'D1': 768, 'D2t/Gallery': 377, 'D2t/Probe': 370}


## 7. Verifica

In [14]:
def summarize(folder):
    files = [os.path.join(dp, f) for dp, _, fs in os.walk(folder) for f in fs if f.endswith('.pt')]
    graphs = [torch.load(p, weights_only=False) for p in files]
    ids = {g.subject_id for g in graphs}
    recs = {g.rec_id for g in graphs}
    shapes = {tuple(g.x.shape) for g in graphs}
    return len(graphs), ids, recs, shapes

for name in ['D1', 'D2t/Gallery', 'D2t/Probe']:
    n_g, ids, recs, shapes = summarize(os.path.join(root, name))
    print(f'{name:12s}: {n_g:5d} grafi | {len(ids):2d} identità | {len(recs):3d} registrazioni | x shapes {shapes}')

ids_d1 = summarize(os.path.join(root, 'D1'))[1]
ids_te = summarize(os.path.join(root, 'D2t'))[1]
assert not (ids_d1 & ids_te), 'Identità in comune fra D1 e D2t!'
print('Nessuna identità in comune fra D1 e D2t  OK')

sample = torch.load(os.path.join(root, 'D2t', 'Probe', sorted(os.listdir(os.path.join(root, 'D2t', 'Probe')))[0]), weights_only=False)
print('\nEsempio di grafo:', sample)
print('subject_id:', sample.subject_id, '| action:', sample.action, '| rec_id:', sample.rec_id, '| win:', sample.win)

D1          :   768 grafi | 22 identità | 220 registrazioni | x shapes {(52, 100)}
D2t/Gallery :   377 grafi | 22 identità | 110 registrazioni | x shapes {(52, 100)}
D2t/Probe   :   370 grafi | 22 identità | 110 registrazioni | x shapes {(52, 100)}
Nessuna identità in comune fra D1 e D2t  OK

Esempio di grafo: Data(x=[52, 100], edge_index=[2, 2704], subject_id='soggetto15', action='tpose', rec_id='soggetto15_tPose1', rec_num=1, win=0)
subject_id: soggetto15 | action: tpose | rec_id: soggetto15_tPose1 | win: 0
